# MISCFilter Deblurring

## 1. Environment Setup & GitHub Sync

In [ ]:
import os

# 請將下方的網址替換為您個人的 GitHub 專案庫網址
REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/MISCFilter.git"

# 防呆機制：如果當前已經在 MISCFilter 目錄內，先退回母目錄
if os.getcwd().endswith('MISCFilter'):
    print("偵測到當前位於 MISCFilter 目錄內，正在自動退回母目錄...")
    %cd ..

# 開始判斷專案資料夾是否存在
if not os.path.exists('MISCFilter'):
    print("首次執行，正在從 GitHub 複製 MISCFilter 專案庫...")
    !git clone {REPO_URL}
    %cd MISCFilter
else:
    %cd MISCFilter
    print("專案已存在！正在強制同步 GitHub 最新程式碼...")
    !git fetch --all
    # 強制完全以 GitHub 上的最新代碼為準，防止 pull 衝突
    !git reset --hard origin/master || git reset --hard origin/main
    !git pull

print(f"Working Directory: {os.getcwd()}")

## 2. Install Dependencies & CuPy

In [ ]:
import torch
import sys

print("正在檢查並補齊 MISCFilter 必備依賴套件...")
!pip install yacs joblib natsort kornia ptflops gdown opencv-python scikit-image matplotlib einops scipy tqdm tensorboard

if os.path.exists('pytorch-gradual-warmup-lr'):
    !pip install ./pytorch-gradual-warmup-lr
else:
    !pip install git+https://github.com/ildoonet/pytorch-gradual-warmup-lr.git

cuda_version = torch.version.cuda
print(f"CUDA Version: {cuda_version}")
if cuda_version:
    major = cuda_version.split('.')[0]
    if major == '12':
        !pip install cupy-cuda12x
    elif major == '11':
        !pip install cupy-cuda11x
    else:
        !pip install cupy
else:
    print("Error: GPU acceleration not active. Enable GPU in settings.")

## 3. Verify Environment & GPU Config

In [ ]:
print("進行環境最終驗證...")
print("PyTorch 版本:", torch.__version__)
print("GPU 是否可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("當前 GPU 裝置:", torch.cuda.get_device_name(0))

# 設定環境變數防止 VRAM 碎片化
%env PYTORCH_ALLOC_CONF=expandable_segments:True

## 4. Setup Pretrained Weights

In [ ]:
WEIGHT_FILE = "RealBlur_J.pth"
WEIGHTS_PATH = ""

if os.path.exists(WEIGHT_FILE):
    WEIGHTS_PATH = f"./{WEIGHT_FILE}"
    print(f"Using local root weights: {WEIGHTS_PATH}")
elif os.path.exists(os.path.join("checkpoints", WEIGHT_FILE)):
    WEIGHTS_PATH = f"./checkpoints/{WEIGHT_FILE}"
    print(f"Using local checkpoint weights: {WEIGHTS_PATH}")
elif os.path.exists("checkpoints") and len(os.listdir("checkpoints")) > 0:
    available_weights = [f for f in os.listdir("checkpoints") if f.endswith('.pth')]
    if available_weights:
        WEIGHTS_PATH = f"./checkpoints/{available_weights[0]}"
        print(f"Using local fallback weights: {WEIGHTS_PATH}")

if not WEIGHTS_PATH:
    print("Weights not found locally. Downloading from Google Drive...")
    os.makedirs('checkpoints', exist_ok=True)
    !pip install --upgrade gdown
    !gdown --folder 1M-Sc_u97vTQskfO6VMzPghb8LnerP8Id -O checkpoints/
    
    if os.path.exists(os.path.join("checkpoints", WEIGHT_FILE)):
        WEIGHTS_PATH = f"./checkpoints/{WEIGHT_FILE}"
    else:
        WEIGHTS_PATH = "./checkpoints/GoPro.pth"

print(f"Final weights path: {WEIGHTS_PATH}")

## 5. Run Deblurring

In [ ]:
os.makedirs('test_images', exist_ok=True)

!python inference_custom.py \
    --input_dir ./test_images \
    --output_dir ./results \
    --weights {WEIGHTS_PATH} \
    --win_size 256 \
    --chunk_size 8 \
    --inference_mode normal \
    --limit 3

## 6. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os

blurred_dir = './test_images'
restored_dir = './results'

restored_files = [f for f in sorted(os.listdir(restored_dir)) if '_deblurred' in f]
if restored_files:
    print(f"Found {len(restored_files)} restored images. Displaying comparisons...")
    for restored_name in restored_files:
        original_name = restored_name.replace('_deblurred', '')
        original_path = os.path.join(blurred_dir, original_name)
        
        if os.path.exists(original_path):
            img_blur = Image.open(original_path)
            img_restore = Image.open(os.path.join(restored_dir, restored_name))
            
            fig, axes = plt.subplots(1, 2, figsize=(18, 9))
            axes[0].imshow(img_blur)
            axes[0].set_title(f'Blurred ({original_name})')
            axes[0].axis('off')
            
            axes[1].imshow(img_restore)
            axes[1].set_title(f'Deblurred ({restored_name})')
            axes[1].axis('off')
            plt.tight_layout()
            plt.show()
        else:
            print(f"Matching original file '{original_name}' not found.")
else:
    print("No results found in ./results.")

## 7. Zip & Download

In [ ]:
!zip -r deblurred_results.zip results/

try:
    from google.colab import files
    files.download('deblurred_results.zip')
except ImportError:
    print("Finished. Download 'deblurred_results.zip' using your browser sidebar.")